# Signed Hertz--Knudsen interfacial flux

## Prerequisites

Read the [kinetic-theory derivation](../../../../docs/lecture-notes/thermal-transport/phase-change/hertz-knudsen/index.md). Review ideal-gas pressure, the Maxwell distribution of one velocity component, and numerical quadrature.

## Objectives

1. Recover the one-way molecular flux for a three-dimensional gas at a planar interface by numerically integrating the positive normal-velocity Maxwell factor after retaining the two normalized tangential factors.
2. Compare that computational derivation with the package-owned Hertz--Knudsen evaluator.
3. Compose a synthetic Antoine equilibrium-pressure model with separate evaporation and condensation coefficients.
4. Plot evaporation, condensation, and signed net flux without silently clamping condensation-dominated states.

Here “first principles” means direct evaluation from the stated classical ideal-gas kinetic-theory assumptions. It does not mean an electronic-structure or many-body first-principles calculation.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad

from projectkoios.physkit.constants import SI
from projectkoios.physkit.thermal.thermo.vaporpressure.antoine import (
    AntoineTemperatureRange,
    AntoineVaporPressureModel,
)
from projectkoios.physkit.thermal.transport.phase_change.hertz_knudsen3d import (
    PlanarHertzKnudsen3DNetFluxModel,
)
from projectkoios.physkit.units import (
    PhysicalUnit,
    ScalarQuantity,
    Unitless,
    VectorQuantity,
)

## One-way flux from a three-dimensional gas

For a planar interface normal to $z$, the three-dimensional Maxwell velocity density factorizes as

$$
f(\mathbf{v})=f_x(v_x)f_y(v_y)f_z(v_z).
$$

The incident one-way number flux is

$$
J_+=n\int_{v_z>0}v_zf(\mathbf{v})\,d^3v.
$$

Both tangential integrals equal one, leaving

$$
J_+=n\int_0^\infty v_z
\sqrt{\frac{m}{2\pi k_BT}}
\exp\!\left(-\frac{mv_z^2}{2k_BT}\right)dv_z.
$$

Using $n=P/(k_BT)$ gives $J_+=P/\sqrt{2\pi mk_BT}$. The numerical exercise evaluates the remaining half-space velocity integral rather than inserting its analytical value. The gas remains three-dimensional; only the normalized tangential integrations have been completed analytically.


In [ ]:
particle_mass = ScalarQuantity(
    magnitude=4.65e-26,
    unit=PhysicalUnit(expression="kilogram"),
)
reference_temperature = ScalarQuantity(
    magnitude=500.0,
    unit=PhysicalUnit(expression="kelvin"),
)
reference_pressure = ScalarQuantity(
    magnitude=200.0,
    unit=PhysicalUnit(expression="pascal"),
)
thermal_velocity_scale = math.sqrt(
    2.0 * SI.k_B * reference_temperature.magnitude / particle_mass.magnitude
)
positive_half_moment, quadrature_error = quad(
    lambda dimensionless_velocity: (
        thermal_velocity_scale
        * dimensionless_velocity
        * np.exp(-(dimensionless_velocity**2))
        / math.sqrt(math.pi)
    ),
    0.0,
    np.inf,
)
number_density = reference_pressure.magnitude / (
    SI.k_B * reference_temperature.magnitude
)
quadrature_number_flux = number_density * positive_half_moment

first_principles_model = PlanarHertzKnudsen3DNetFluxModel(
    particle_mass=particle_mass,
    evaporation_coefficient=ScalarQuantity(magnitude=1.0, unit=Unitless()),
    condensation_coefficient=ScalarQuantity(magnitude=0.0, unit=Unitless()),
)
first_principles_evaluation = first_principles_model.evaluate(
    temperatures=VectorQuantity(
        magnitude=np.array([reference_temperature.magnitude], dtype=np.float64),
        unit=reference_temperature.unit,
    ),
    equilibrium_pressures=VectorQuantity(
        magnitude=np.array([reference_pressure.magnitude], dtype=np.float64),
        unit=reference_pressure.unit,
    ),
    ambient_pressures=VectorQuantity(
        magnitude=np.array([0.0], dtype=np.float64),
        unit=reference_pressure.unit,
    ),
    number_flux_unit=PhysicalUnit(expression="meter ** -2 / second"),
    mass_flux_unit=PhysicalUnit(expression="kilogram / meter ** 2 / second"),
)
quadrature_relative_tolerance = 2.0e-14
assert np.isclose(
    quadrature_number_flux,
    first_principles_evaluation.evaporation_number_flux.magnitude[0],
    rtol=quadrature_relative_tolerance,
    atol=0.0,
)
quadrature_error_relative_limit = 4.0e-9
assert quadrature_error / positive_half_moment < quadrature_error_relative_limit

In [ ]:
temperature_unit = PhysicalUnit(expression="kelvin")
pressure_unit = PhysicalUnit(expression="pascal")
minimum_temperature_kelvin = 300.0
maximum_temperature_kelvin = 800.0
temperature_sample_count = 500
temperature_values_kelvin = np.linspace(
    minimum_temperature_kelvin,
    maximum_temperature_kelvin,
    temperature_sample_count,
    dtype=np.float64,
)
temperatures = VectorQuantity(
    magnitude=temperature_values_kelvin,
    unit=temperature_unit,
)
synthetic_vapor_pressure_model = AntoineVaporPressureModel(
    A=ScalarQuantity(magnitude=5.0, unit=Unitless()),
    B=ScalarQuantity(magnitude=1000.0, unit=temperature_unit),
    C=ScalarQuantity(magnitude=0.0, unit=temperature_unit),
    coefficient_temperature_unit=temperature_unit,
    coefficient_pressure_unit=pressure_unit,
    valid_temperature_range=AntoineTemperatureRange(
        lower=ScalarQuantity(
            magnitude=minimum_temperature_kelvin,
            unit=temperature_unit,
        ),
        upper=ScalarQuantity(
            magnitude=maximum_temperature_kelvin,
            unit=temperature_unit,
        ),
    ),
    source="Synthetic illustrative coefficients; not fitted material data.",
)
equilibrium_pressure_evaluation = synthetic_vapor_pressure_model.evaluate(
    temperatures=temperatures,
    pressure_unit=pressure_unit,
)
ambient_pressure_pascal = 100.0
ambient_pressures = VectorQuantity(
    magnitude=np.full(
        temperature_sample_count,
        ambient_pressure_pascal,
        dtype=np.float64,
    ),
    unit=pressure_unit,
)

In [ ]:
net_flux_model = PlanarHertzKnudsen3DNetFluxModel(
    particle_mass=particle_mass,
    evaporation_coefficient=ScalarQuantity(magnitude=0.8, unit=Unitless()),
    condensation_coefficient=ScalarQuantity(magnitude=0.4, unit=Unitless()),
)
flux_evaluation = net_flux_model.evaluate(
    temperatures=temperatures,
    equilibrium_pressures=equilibrium_pressure_evaluation.pressures,
    ambient_pressures=ambient_pressures,
    number_flux_unit=PhysicalUnit(expression="meter ** -2 / second"),
    mass_flux_unit=PhysicalUnit(expression="kilogram / meter ** 2 / second"),
)
assert np.allclose(
    flux_evaluation.net_number_flux.magnitude,
    flux_evaluation.evaporation_number_flux.magnitude
    - flux_evaluation.condensation_number_flux.magnitude,
)
assert np.any(flux_evaluation.net_number_flux.magnitude < 0.0)
assert np.any(flux_evaluation.net_number_flux.magnitude > 0.0)

In [ ]:
figure_size_inches = (12.0, 4.8)
figure, (pressure_axis, flux_axis) = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=figure_size_inches,
)
pressure_axis.semilogy(
    temperatures.magnitude,
    equilibrium_pressure_evaluation.pressures.magnitude,
    label="synthetic equilibrium pressure",
)
pressure_axis.semilogy(
    temperatures.magnitude,
    ambient_pressures.magnitude,
    linestyle="--",
    label="ambient pressure",
)
pressure_axis.set(
    xlabel="temperature (K)",
    ylabel="pressure (Pa)",
    title="Pressure inputs",
)
pressure_axis.grid(True)
pressure_axis.legend()

flux_axis.plot(
    temperatures.magnitude,
    flux_evaluation.evaporation_number_flux.magnitude,
    label="evaporation contribution",
)
flux_axis.plot(
    temperatures.magnitude,
    -flux_evaluation.condensation_number_flux.magnitude,
    label="negative condensation contribution",
)
flux_axis.plot(
    temperatures.magnitude,
    flux_evaluation.net_number_flux.magnitude,
    linewidth=2.0,
    label="signed net flux",
)
flux_axis.axhline(0.0, color="black", linewidth=0.8, linestyle=":")
flux_axis.set(
    xlabel="temperature (K)",
    ylabel="number flux (m⁻² s⁻¹)",
    title="Separate and signed Hertz–Knudsen fluxes",
)
flux_axis.grid(True)
flux_axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

The numerical half-Maxwellian integral agrees with the analytical package evaluator under the same classical ideal-gas assumptions. In the synthetic temperature sweep, the equilibrium pressure increases while the ambient pressure remains fixed. Separate transfer coefficients weight the two one-way contributions, and the signed net flux changes sign when condensation overtakes evaporation or vice versa.

A negative net flux is retained as condensation toward the condensed phase; it is not silently replaced by zero. The synthetic Antoine coefficients, particle mass, ambient pressure, and transfer coefficients are illustrative and are not material validation data.

## Exercises

1. Derive the positive half-space velocity moment analytically.
2. Change the two transfer coefficients independently and predict the zero crossing.
3. Convert the requested flux units and verify that the physical result is unchanged.
4. Add a separately specified surface-temperature or nonequilibrium-gas model and identify which derivation assumptions no longer hold.
